# PanDerm LoRA + Multi-Layer Fusion Fine-tuning — Oral_Diseases

**목적**: `docs/eda/panderm_domain_generalization_strategy.md`의 **Tier 2 조합** — **축 B4 (LoRA, PEFT)** + **축 A3 (여러 레이어 CLS 융합)** — 를 실제로 구현해, few-shot(클래스당 ~100 이하) 인접 도메인 시나리오에서 현재 frozen linear-probe baseline을 개선할 수 있는지 검증한다.

**현재 baseline (frozen CLS + linear probe, `PanDerm/output_dir/oral_diseases_panderm_large_lp/`)**:

| W_F1 | AUROC | BACC | ACC | AUPR |
|---|---|---|---|---|
| 0.7584 | 0.9463 | 0.8107 | 0.7778 | 0.7981 |

**이번 실험이 다른 점**:
- 마지막 CLS 1개(1024-d) 대신, **여러 레이어의 CLS를 concat**(A3)해서 head에 넣는다.
- 선형 head만 학습하는 대신, **attention block에 LoRA 어댑터**(B4)를 붙여 backbone도 아주 적은 파라미터로 같이 적응시킨다.

**RTX 3070 (8GB, 실행 시점 여유 메모리 ~3.6GB) 대응**:
- **Gradient checkpointing** — 블록별로 activation을 저장하지 않고 backward 시 재계산 → 활성화 메모리가 24개 블록 합산이 아니라 블록 1개 수준으로 줄어듦. 이게 8GB 카드에서 ViT-L을 학습 가능하게 하는 핵심 장치.
- **AMP(fp16 autocast)** — 연산/activation을 fp16으로.
- **Backbone은 거의 전부 frozen** — LoRA 어댑터(<1% 파라미터)와 커스텀 head만 학습 → optimizer state(Adam m/v)가 거의 안 붙음.
- **작은 배치 + gradient accumulation** — `batch_size=8 × accum_steps=4` = 유효배치 32.
- `peft` 라이브러리는 설치되어 있지 않고(확인함), PanDerm의 `Attention.forward`가 `self.qkv(x)`가 아니라 `F.linear(weight=self.qkv.weight, ...)`로 **직접 weight에 접근**하는 방식이라 `nn.Module` 래핑만으로는 LoRA가 걸리지 않는다. 그래서 `Attention.forward` 자체를 monkey-patch하는 방식으로 직접 구현한다(아래 참고).

**참고 코드 위치**: `PanDerm/classification/models/modeling_finetune.py` — `VisionTransformer.forward_features:496`, `Attention.forward:170`, `Block.forward:285`.

## 0. 환경 설정 & 경로

In [1]:
import os, sys, math, types, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from torchvision import transforms

from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, roc_auc_score,
    average_precision_score, classification_report,
)
from sklearn.preprocessing import label_binarize

# ── 경로 설정 (기존 notebooks/PanDerm/Linear Evaluation/*.ipynb 관례와 동일)
NOTEBOOK_DIR  = Path(os.getcwd())
PROJECT_ROOT  = (NOTEBOOK_DIR / "../../..").resolve()
CLASSIFICATION_DIR = PROJECT_ROOT / "PanDerm" / "classification"
if str(CLASSIFICATION_DIR) not in sys.path:
    sys.path.insert(0, str(CLASSIFICATION_DIR))

DATA_ROOT   = PROJECT_ROOT / "PanDerm" / "data"
OUTPUT_ROOT = PROJECT_ROOT / "PanDerm" / "output_dir"
CHECKPOINT  = PROJECT_ROOT / "PanDerm" / "checkpoint" / "panderm_ll_data6_checkpoint-499.pth"

DATASET_NAME = "Oral_Diseases"
META_CSV     = DATA_ROOT / "Oral_Diseases" / "Linear Evaluation" / "oral_diseases_multiclass.csv"
IMAGE_ROOT   = str(DATA_ROOT / "Oral_Diseases") + "/"   # Derm_Dataset이 root+filename을 단순 문자열 접합
CLASS_NAMES  = ["CaS", "CoS", "Gum", "MC", "OC", "OLP", "OT"]
NUM_CLASSES  = 7

RESULT_DIR = OUTPUT_ROOT / "oral_diseases_lora_multilayer_lp"
os.makedirs(RESULT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"free={free_b/1024**3:.2f}GB / total={total_b/1024**3:.2f}GB")
    if free_b / 1024**3 < 2.0:
        print("[경고] 여유 GPU 메모리가 2GB 미만입니다. 다른 프로세스를 정리하거나 "
              "아래 BATCH_SIZE / FUSION_LAYERS 개수를 줄이세요.")

def gpu_mem_report(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2
        print(f"[GPU mem: {tag}] allocated={alloc:.0f}MB reserved={reserved:.0f}MB")

device: cuda
GPU: NVIDIA GeForce RTX 3070
free=6.93GB / total=8.00GB


**결과 해석**: `device: cuda`와 GPU 이름이 나오면 정상입니다. `free=...GB`가 실행 시점의 실제 여유 메모리이니, 이 값이 낮게 나오면(예: 2GB 미만) 아래 학습 설정(배치 크기, 융합 레이어 수)을 더 보수적으로 잡아야 합니다.

## 1. 데이터셋

`Oral_Diseases` — 7-class, train 407 / val 55 / test 54 (클래스당 42~74장, few-shot 시나리오). 기존 `Derm_Dataset`(`PanDerm/classification/datasets/derm_data.py`)을 그대로 재사용하되, **train에는 약한 증강**(few-shot 과적합 완화)을, val/test에는 기존 baseline과 동일한 `Resize(256)+CenterCrop(224)` 평가 변환을 적용한다.

In [2]:
from datasets.derm_data import Derm_Dataset
from models.builder import get_norm_constants, get_eval_transforms

NORM_MEAN, NORM_STD = get_norm_constants("imagenet")   # baseline과 동일 정규화 상수 사용

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])
eval_transform = get_eval_transforms(which_img_norm="imagenet")   # Resize(256)+CenterCrop(224), baseline과 동일

df_all = pd.read_csv(META_CSV)

dataset_train = Derm_Dataset(df=df_all, root=IMAGE_ROOT, train=True, transforms=train_transform, binary=False)
dataset_val   = Derm_Dataset(df=df_all, root=IMAGE_ROOT, val=True,   transforms=eval_transform,  binary=False)
dataset_test  = Derm_Dataset(df=df_all, root=IMAGE_ROOT, test=True,  transforms=eval_transform,  binary=False)

print(f"train={len(dataset_train)}  val={len(dataset_val)}  test={len(dataset_test)}")

BATCH_SIZE = 8     # RTX3070 8GB 기준 학습 배치. OOM 시 4로 낮출 것.
EVAL_BATCH_SIZE = 32   # 평가는 backward가 없어 훨씬 가벼움

train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,
                                            num_workers=4, pin_memory=True, drop_last=True)
val_loader   = torch.utils.data.DataLoader(dataset_val, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                                            num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(dataset_test, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                                            num_workers=2, pin_memory=True)

train_label_counts = df_all[df_all["split"] == "train"]["label"].value_counts().sort_index().values
print("train 클래스별 개수:", dict(zip(CLASS_NAMES, train_label_counts)))

# 현재 파이프라인의 알려진 결함(class weight 미적용) 보완 — 축 B1
class_weights = torch.tensor(
    train_label_counts.sum() / (len(train_label_counts) * train_label_counts), dtype=torch.float32
).to(device)
print("class weights:", class_weights.cpu().numpy().round(3))

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


normalization method:  imagenet
normalization method:  imagenet
train=407  val=55  test=54
train 클래스별 개수: {'CaS': np.int64(63), 'CoS': np.int64(59), 'Gum': np.int64(47), 'MC': np.int64(72), 'OC': np.int64(42), 'OLP': np.int64(74), 'OT': np.int64(50)}
class weights: [0.923 0.985 1.237 0.808 1.384 0.786 1.163]


**결과 해석**: `drop_last=True`는 배치 정규화/체크포인팅 재계산 시 배치 크기가 들쭉날쭉하지 않도록 마지막 미니배치를 버리는 옵션입니다(407장 기준 영향은 미미). `class weights`는 표본이 적은 클래스(OC=42장)에 더 큰 가중치를, 많은 클래스(OLP=74장)에 더 작은 가중치를 줘서 few-shot 불균형을 완화합니다.

## 2. 모델 정의 — LoRA 어댑터 + 여러 레이어 CLS 융합

**LoRA 주입 방식에 대한 주의**: `Attention.forward`는 `self.qkv(x)`가 아니라 `F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)`로 **weight 텐서에 직접 접근**한다(q/v에만 별도 bias를 더하는 PanDerm 고유 구현 때문). 따라서 `self.qkv`를 다른 모듈로 치환해도 `forward`는 여전히 원본 weight만 사용해 LoRA가 무시된다. 해결책: `Attention.forward` 자체를 LoRA 델타를 더하는 버전으로 **monkey-patch**한다(`types.MethodType`).

In [3]:
class LoRALayer(nn.Module):
    """저랭크 어댑터: out = (x @ A^T @ B^T) * (alpha/r). B는 0으로 초기화해 학습 시작 시 델타=0."""
    def __init__(self, in_dim, out_dim, r=8, alpha=16):
        super().__init__()
        self.A = nn.Parameter(torch.zeros(r, in_dim))
        self.B = nn.Parameter(torch.zeros(out_dim, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        return (x @ self.A.t() @ self.B.t()) * self.scale


def _lora_attention_forward(self, x, rel_pos_bias=None):
    """원본 Attention.forward(modeling_finetune.py:170)에 lora_qkv/lora_proj 델타만 추가한 버전.
    types.MethodType으로 바인딩되므로 self는 Attention 인스턴스."""
    B, N, C = x.shape
    qkv_bias = None
    if self.q_bias is not None:
        qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias, requires_grad=False), self.v_bias))
    qkv = F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)
    qkv = qkv + self.lora_qkv(x)
    qkv = qkv.reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

    q = q * self.scale
    attn = q @ k.transpose(-2, -1)

    if self.relative_position_bias_table is not None:
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size[0] * self.window_size[1] + 1,
               self.window_size[0] * self.window_size[1] + 1, -1)
        attn = attn + relative_position_bias.permute(2, 0, 1).contiguous().unsqueeze(0)
    if rel_pos_bias is not None:
        attn = attn + rel_pos_bias

    attn = attn.softmax(dim=-1)
    attn = self.attn_drop(attn)

    out = (attn @ v).transpose(1, 2).reshape(B, N, -1)
    out = self.proj(out) + self.lora_proj(out)
    out = self.proj_drop(out)
    return out


def inject_lora(backbone, layer_indices, r=8, alpha=16):
    """지정한 블록들의 attn.qkv / attn.proj에 LoRA를 붙이고 forward를 patch. 새로 추가된 파라미터 리스트 반환."""
    dim = backbone.embed_dim
    lora_params = []
    for i in layer_indices:
        attn = backbone.blocks[i].attn
        qkv_out_dim = attn.qkv.weight.shape[0]     # all_head_dim * 3
        proj_in_dim = attn.proj.weight.shape[1]    # all_head_dim
        attn.lora_qkv = LoRALayer(dim, qkv_out_dim, r=r, alpha=alpha)
        attn.lora_proj = LoRALayer(proj_in_dim, dim, r=r, alpha=alpha)
        attn.forward = types.MethodType(_lora_attention_forward, attn)
        lora_params += list(attn.lora_qkv.parameters()) + list(attn.lora_proj.parameters())
    return lora_params

**결과 해석**: 이 셀은 클래스/함수 정의만 하므로 출력은 없습니다. `LoRALayer`의 `B`를 0으로 초기화하는 게 핵심 트릭 — 학습 시작 시점엔 LoRA 델타가 정확히 0이라, 사전학습된 PanDerm의 원래 동작에서 출발해 서서히 적응합니다.

In [4]:
class PanDermLoRAMultiLayer(nn.Module):
    """frozen PanDerm 백본 + 지정 블록 LoRA + 여러 레이어 CLS 융합(A3) + 커스텀 head."""

    def __init__(self, backbone, fusion_layers, num_classes,
                 lora_r=8, lora_alpha=16, lora_layers=None,
                 use_grad_checkpoint=True, head_dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.fusion_layers = sorted(fusion_layers)
        self.max_layer = self.fusion_layers[-1]
        self.use_grad_checkpoint = use_grad_checkpoint
        embed_dim = backbone.embed_dim

        for p in self.backbone.parameters():
            p.requires_grad = False

        target_layers = lora_layers if lora_layers is not None else self.fusion_layers
        self.lora_params = inject_lora(self.backbone, target_layers, r=lora_r, alpha=lora_alpha)
        for p in self.lora_params:
            p.requires_grad = True

        # 블록 출력은 최종 LayerNorm 적용 이전이라 레이어마다 스케일이 다름 → 레이어별 학습 가능한 LN으로 보정
        self.layer_norms = nn.ModuleList([nn.LayerNorm(embed_dim, eps=1e-6) for _ in self.fusion_layers])
        self.dropout = nn.Dropout(head_dropout)
        self.head = nn.Linear(embed_dim * len(self.fusion_layers), num_classes)

        self.backbone.eval()   # dropout/drop_path 노이즈 비활성화 (항상 eval 유지)

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()   # backbone은 절대 train() 모드로 안 바뀌게 오버라이드
        return self

    def forward(self, x):
        bb = self.backbone
        x = bb.patch_embed(x)
        B = x.shape[0]
        cls_tokens = bb.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + bb.pos_embed.expand(B, -1, -1).type_as(x).to(x.device).detach()
        x = bb.pos_drop(x)
        rel_pos_bias = bb.rel_pos_bias() if bb.rel_pos_bias is not None else None

        collected = {}
        for i in range(self.max_layer + 1):     # max_layer 이후 블록은 실행할 필요 없음 (조기 종료)
            blk = bb.blocks[i]
            if self.use_grad_checkpoint and self.training:
                x = grad_checkpoint(blk, x, rel_pos_bias, use_reentrant=False)
            else:
                x = blk(x, rel_pos_bias=rel_pos_bias)
            if i in self.fusion_layers:
                collected[i] = x[:, 0]

        fused = torch.cat([ln(collected[i]) for ln, i in zip(self.layer_norms, self.fusion_layers)], dim=1)
        fused = self.dropout(fused)
        return self.head(fused)

    def trainable_parameter_groups(self, head_lr=1e-3, lora_lr=1e-4, weight_decay=1e-4):
        lora_and_ln = list(self.layer_norms.parameters()) + self.lora_params
        return [
            {"params": self.head.parameters(), "lr": head_lr, "weight_decay": weight_decay},
            {"params": lora_and_ln, "lr": lora_lr, "weight_decay": weight_decay},
        ]

    def trainable_param_names(self):
        return {n for n, p in self.named_parameters() if p.requires_grad}

    def trainable_state_dict(self):
        names = self.trainable_param_names()
        return {k: v.detach().cpu().clone() for k, v in self.state_dict().items() if k in names}

    def load_trainable_state_dict(self, sd):
        self.load_state_dict(sd, strict=False)

**결과 해석**: 클래스 정의 셀이라 출력 없음. `forward`에서 `range(self.max_layer + 1)`로 마지막 융합 레이어 이후 블록은 아예 돌리지 않는 것이 메모리/연산 절약 포인트입니다(예: 마지막 융합 레이어를 23이 아니라 19로 두면 블록 20~23을 건너뜁니다). 지금 기본 설정은 baseline과 비교하기 위해 마지막 블록(23)을 포함합니다.

## 3. 모델 생성 & 파라미터 확인

In [5]:
from models.builder import get_encoder
import argparse

FUSION_LAYERS = [11, 15, 19, 23]     # 0~23 중 4개 레이어의 CLS를 융합 (A3)
LORA_LAYERS   = list(range(24))      # 모든 블록에 LoRA 적용 (파라미터 비용이 작아 전부 적용해도 무방)
LORA_R, LORA_ALPHA = 8, 16
USE_GRAD_CHECKPOINT = True

args = argparse.Namespace(pretrained_checkpoint=str(CHECKPOINT))
backbone, _ = get_encoder(args, "PanDerm_Large_LP")   # ViT-L/16, ImageNet 사전학습 아님 — PanDerm SSL 체크포인트

model = PanDermLoRAMultiLayer(
    backbone, FUSION_LAYERS, NUM_CLASSES,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_layers=LORA_LAYERS,
    use_grad_checkpoint=USE_GRAD_CHECKPOINT,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable params: {trainable:,}")
print(f"frozen params   : {frozen:,}")
print(f"trainable ratio : {100 * trainable / (trainable + frozen):.3f}%")
gpu_mem_report("모델 로드 후")

loading model checkpoint


/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      

**결과 해석**: `trainable ratio`가 1% 훨씬 아래로 나오면 정상입니다(LoRA r=8을 24블록에 적용해도 수십만 파라미터 수준, 3억 파라미터 backbone 대비 미미). 이 비율이 낮을수록 few-shot에서 과적합 위험이 낮다는 뜻입니다. `모델 로드 후` GPU 메모리는 backbone 가중치(fp32, ~1.2GB) 위주로 잡힙니다.

## 4. 메모리 sanity check (RTX 3070)

본 학습을 시작하기 전에 **1 스텝만** 먼저 돌려서 OOM 여부를 확인합니다. 여기서 문제 없으면 전체 학습 루프도 안전합니다.

In [6]:
model.train()
sample_images, sample_labels, _ = next(iter(train_loader))
sample_images, sample_labels = sample_images.to(device), sample_labels.to(device)

scaler_test = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
optimizer_test = torch.optim.AdamW(model.trainable_parameter_groups())

with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
    logits = model(sample_images)
    loss = F.cross_entropy(logits, sample_labels)
scaler_test.scale(loss).backward()
gpu_mem_report("1 스텝 forward+backward 후 (sanity check)")
optimizer_test.zero_grad(set_to_none=True)
del sample_images, sample_labels, logits, loss, optimizer_test, scaler_test
torch.cuda.empty_cache()
gpu_mem_report("empty_cache 후")

[GPU mem: 1 스텝 forward+backward 후 (sanity check)] allocated=1192MB reserved=1526MB
[GPU mem: empty_cache 후] allocated=1183MB reserved=1224MB


**결과 해석**: `reserved` 값이 GPU 여유 메모리(위 0번 셀에서 확인한 `free`)보다 한참 작으면 안전합니다. 만약 여기서 OOM이 나면:
1. `BATCH_SIZE`를 8→4로 낮추거나,
2. `FUSION_LAYERS`에서 마지막 레이어(23)를 빼서(예: `[9, 13, 17]`) 조기 종료 구간을 늘리거나,
3. `LORA_LAYERS`를 `list(range(12, 24))`처럼 절반만 적용하세요(연산 안 줄지만 LoRA 자체 메모리는 더 줄어듦 — 다만 가장 큰 레버는 1, 2번).

## 5. 학습 루프

In [7]:
ACCUM_STEPS = 4          # 유효 배치 = BATCH_SIZE * ACCUM_STEPS = 32
EPOCHS = 40
HEAD_LR, LORA_LR, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
EARLY_STOP_PATIENCE = 10
USE_AMP = torch.cuda.is_available()

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.trainable_parameter_groups(HEAD_LR, LORA_LR, WEIGHT_DECAY))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_val_bacc = -1.0
best_state = None
patience_counter = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    for step, (images, labels, _) in enumerate(train_loader):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels) / ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images = images.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
            val_preds.append(logits.argmax(1).cpu())
            val_targets.append(labels)
    val_preds = torch.cat(val_preds).numpy()
    val_targets = torch.cat(val_targets).numpy()
    val_bacc = balanced_accuracy_score(val_targets, val_preds)
    history.append({"epoch": epoch + 1, "train_loss": running_loss / len(train_loader), "val_bacc": val_bacc})

    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={running_loss/len(train_loader):.4f}  val_bacc={val_bacc:.4f}")

    if val_bacc > best_val_bacc:
        best_val_bacc = val_bacc
        best_state = model.trainable_state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"[early stop] epoch {epoch+1} (best val_bacc={best_val_bacc:.4f})")
            break

gpu_mem_report("학습 종료 후")
print(f"best val_bacc = {best_val_bacc:.4f}")

epoch 01/40  train_loss=1.6496  val_bacc=0.6713
epoch 02/40  train_loss=0.9331  val_bacc=0.7112
epoch 03/40  train_loss=0.6657  val_bacc=0.6617
epoch 04/40  train_loss=0.5015  val_bacc=0.6359
epoch 05/40  train_loss=0.4504  val_bacc=0.6803
epoch 06/40  train_loss=0.3725  val_bacc=0.6543
epoch 07/40  train_loss=0.3303  val_bacc=0.7150
epoch 08/40  train_loss=0.2686  val_bacc=0.6563
epoch 09/40  train_loss=0.2216  val_bacc=0.7025
epoch 10/40  train_loss=0.1665  val_bacc=0.6583
epoch 11/40  train_loss=0.1380  val_bacc=0.6707
epoch 12/40  train_loss=0.1225  val_bacc=0.6991
epoch 13/40  train_loss=0.0949  val_bacc=0.6980
epoch 14/40  train_loss=0.0651  val_bacc=0.6741
epoch 15/40  train_loss=0.0542  val_bacc=0.7184
epoch 16/40  train_loss=0.0442  val_bacc=0.6805
epoch 17/40  train_loss=0.0454  val_bacc=0.7580
epoch 18/40  train_loss=0.0518  val_bacc=0.6980
epoch 19/40  train_loss=0.0588  val_bacc=0.7104
epoch 20/40  train_loss=0.0427  val_bacc=0.7138
epoch 21/40  train_loss=0.0403  val_bacc

**결과 해석**: `val_bacc`가 epoch을 거치며 대체로 상승하다가 정체/하락하면 early stopping이 그 지점에서 멈춥니다. 55장짜리 val set이라 `val_bacc`가 계단식으로(이산적으로) 변하는 게 정상입니다 — 하나의 예측이 바뀔 때마다 값이 뛰기 때문입니다. `best_state`에는 **학습 가능한 파라미터만**(LoRA + LayerNorm + head, backbone 3억 파라미터 제외) 저장되어 있어 체크포인트가 가볍습니다.

## 6. 테스트 평가 & baseline 비교

In [8]:
def compute_metrics(y_true, y_pred, probs, num_classes):
    bacc = balanced_accuracy_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    wf1 = classification_report(y_true, y_pred, output_dict=True, zero_division=0)["weighted avg"]["f1-score"]
    auroc = roc_auc_score(y_true, probs, average="macro", multi_class="ovr")
    y_true_oh = label_binarize(y_true, classes=list(range(num_classes)))
    aupr = average_precision_score(y_true_oh, probs, average="macro")
    return {"bacc": bacc, "acc": acc, "weighted_f1": wf1, "auroc": auroc, "aupr": aupr}


model.load_trainable_state_dict(best_state)
model.eval()

test_preds, test_targets, test_probs = [], [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
        probs = F.softmax(logits.float(), dim=1)
        test_preds.append(probs.argmax(1).cpu())
        test_targets.append(labels)
        test_probs.append(probs.cpu())

test_preds = torch.cat(test_preds).numpy()
test_targets = torch.cat(test_targets).numpy()
test_probs = torch.cat(test_probs).numpy()

lora_metrics = compute_metrics(test_targets, test_preds, test_probs, NUM_CLASSES)

baseline_csv = OUTPUT_ROOT / "oral_diseases_panderm_large_lp" / "oral_diseases_panderm_large_lp_result.csv"
baseline_row = pd.read_csv(baseline_csv).iloc[0]

comparison = pd.DataFrame([
    {"method": "baseline (frozen CLS + linear probe)",
     "bacc": float(baseline_row["BACC"]), "auroc": float(baseline_row["AUROC"]),
     "aupr": float(baseline_row["AUPR"]), "acc": float(baseline_row["ACC"]), "weighted_f1": float(baseline_row["W_F1"])},
    {"method": f"LoRA(r={LORA_R}) + multi-layer fusion {FUSION_LAYERS}",
     "bacc": lora_metrics["bacc"], "auroc": lora_metrics["auroc"],
     "aupr": lora_metrics["aupr"], "acc": lora_metrics["acc"], "weighted_f1": lora_metrics["weighted_f1"]},
])
comparison = comparison.set_index("method").round(4)
print(comparison)

comparison.to_csv(RESULT_DIR / "comparison_vs_baseline.csv")
pd.DataFrame(history).to_csv(RESULT_DIR / "training_history.csv", index=False)
torch.save(best_state, RESULT_DIR / "best_trainable_state_dict.pt")
print(f"\n결과 저장 위치: {RESULT_DIR}")

                                                   bacc   auroc    aupr  \
method                                                                    
baseline (frozen CLS + linear probe)             0.8107  0.9463  0.7981   
LoRA(r=8) + multi-layer fusion [11, 15, 19, 23]  0.8250  0.9630  0.8712   

                                                    acc  weighted_f1  
method                                                                
baseline (frozen CLS + linear probe)             0.7778       0.7584  
LoRA(r=8) + multi-layer fusion [11, 15, 19, 23]  0.7963       0.7875  

결과 저장 위치: /home/junkim/paper_ajou_dev/PanDerm/output_dir/oral_diseases_lora_multilayer_lp


**결과 해석**: 표의 두 행을 비교합니다 — 특히 **`bacc`(균형 정확도)**와 **`aupr`**을 우선 확인하세요(불균형 데이터셋이라 이 둘이 `acc`보다 신뢰도가 높습니다). LoRA+융합 방법이 baseline보다 `bacc`/`aupr`이 높으면 few-shot·인접도메인 설정에서 이 조합이 도움이 된다는 근거가 됩니다. 반대로 별 차이가 없거나 낮다면:
- `FUSION_LAYERS` 조합을 바꿔보기(예: 더 이른 레이어 포함),
- `LORA_R`을 4나 16으로 바꿔보기,
- `EPOCHS`/`EARLY_STOP_PATIENCE`를 늘려보기

등을 시도할 수 있습니다. `RESULT_DIR`에 학습 곡선(`training_history.csv`)과 학습된 파라미터(`best_trainable_state_dict.pt`, backbone 제외라 용량이 작음)가 저장되니 재현·재사용이 가능합니다.

## 다음 단계 제안

1. **레이어 스윕**: `FUSION_LAYERS`를 단일 레이어로 바꿔가며(0~23) test BACC를 스캔 → 어느 레이어가 구강 도메인에 제일 잘 전이되는지 곡선으로 확인 (`docs/eda/panderm_domain_generalization_strategy.md` 축 A3 참고).
2. **LoRA 단독 vs 융합 단독 ablation**: `FUSION_LAYERS=[23]`(현재 baseline과 동일한 레이어, LoRA만 추가)과 `LORA_LAYERS=[]`(LoRA 없이 융합만) 두 변형을 돌려 어느 축이 개선에 더 기여했는지 분리.
3. **DermLIP zero-shot과 비교**(축 D): 라벨 없이 나온 zero-shot 점수를 상한선/하한선 참고치로 같이 보고.
4. 이번 실험이 유의미하면 다른 인접 도메인 데이터셋(Nail_Disease 등, 단 8차 미팅 지시로 제외된 것은 예외)에도 동일 파이프라인 적용.